# Dominio 3: Sector Agrícola (Banco Mundial)

8 indicadores reales del Banco Mundial (`api.worldbank.org`, sin
autenticación) para 9 países sudamericanos, 1990-2025: rendimiento de
cereales (target), fertilizante, tierra arable/agrícola/irrigada, PIB
agrícola, población rural, índice de producción de cultivos.

Este notebook ejecuta el pipeline real de `src/domains/agriculture_worldbank/`.


## 1. Limpieza del panel país×año


In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src.domains.agriculture_worldbank.clean import build_clean_panel

panel, report = build_clean_panel()
for k, v in report.items():
    if not isinstance(v, pd.DataFrame):
        print(f"{k}: {v}")


**Hallazgo real**: la mayoría de los indicadores están casi completos por
país, pero `AG.LND.IRIG.AG.ZS` (tierra agrícola irrigada) es la excepción
seria -- Perú no tiene NI UN SOLO valor real en 36 años. Un hueco así no es
interpolable (no hay ningún punto de anclaje dentro de la propia serie de
Perú); se resuelve con imputación por media entre países, documentado
explícitamente en vez de disfrazarlo de interpolación.


## 2. Features y modelado (>=100 épocas)


In [ ]:
from src.domains.agriculture_worldbank.features import build_features

features_df = build_features(panel)
print(f"{len(features_df):,} filas x features")
features_df.describe().T


In [ ]:
import json
metrics = json.load(open("../outputs/agriculture/metrics.json"))
pd.DataFrame(metrics["results"]).T


**Resultado real**: tanto la MLP (R²=0.87) como XGBoost (R²=0.88) superan
ampliamente al baseline (R²=-0.33, negativo porque el rendimiento de
cereales tiene una tendencia real al alza que un promedio histórico
subestima) -- acá SÍ hay señal real y fuerte (fertilizante, tierra irrigada,
tendencia agronómica), a diferencia del dominio financiero.


## 3. Gráficos


**% de nulos por indicador, antes/después de interpolar** -- muestra el caso extremo de tierra irrigada.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/agriculture/figures/missingness_before_after.png"))


**Correlación entre features y el rendimiento de cereales del año siguiente.**


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/agriculture/figures/feature_correlation.png"))


**Rendimiento de cereales real, Chile, 1990-2025.**


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/agriculture/figures/cereal_yield_timeseries_CHL.png"))


**Rendimiento de cereales real, Argentina, 1990-2025.**


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/agriculture/figures/cereal_yield_timeseries_ARG.png"))


**Curva de entrenamiento de la MLP** -- >=100 épocas, con early stopping real (mejor época marcada).


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/agriculture/figures/mlp_training_curve.png"))


**Rendimiento real vs. predicho (holdout 2020-2025).**


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/agriculture/figures/regression_diagnostics.png"))


**Comparación baseline vs. MLP vs. XGBoost** -- ambos modelos reales le ganan al baseline por un margen amplio.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/agriculture/figures/model_comparison.png"))


## Conclusiones

- El panel agrícola tiene señal real y explotable (fertilizante, irrigación,
  tendencia agronómica) -- contraste honesto con el dominio financiero.
- La missingness no es uniforme: la mayoría de los huecos son bordes
  temporales triviales, pero un caso (Perú/irrigación) requiere imputación
  cross-país explícita, no interpolación.
